# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

# ماژول ساخت سند

In [325]:
import re
import unicodedata

class TextPreprocessor:
    def __init__(self):
        self.stopwords = {
            "the", "is", "in", "at", "of", "and", "a", "on", "was", "were",
            "by", "for", "to", "with", "about", "against", "from", "its",
            "draw", "matches", "matche"
        }

    def normalize_text(self, text: str) -> str:
        if not text:
            return ""
        text = unicodedata.normalize("NFKD", text)
        text = text.encode("ascii", "ignore").decode("ascii")
        text = text.lower()
        text = text.replace('\xa0', ' ')
        text = re.sub(r"&[a-z0-9#]+;", " ", text)


        text = re.sub(r"'s\b", "", text)

        text = re.sub(r"[^a-z0-9\-\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def _to_singular(self, word: str) -> str:

        if word.endswith('ies') and len(word) > 3:
            return word[:-3] + 'y'
        if word.endswith('ses') or word.endswith('xes') or word.endswith('ches') or word.endswith('shes'):
            return word[:-1]
        if word.endswith('s') and len(word) > 3:
            return word[:-1]
        return word

    def tokenize(self, text: str) -> list:
        base_tokens = text.split()
        expanded_tokens = []
        for token in base_tokens:
            if "-" in token:
                if re.fullmatch(r"\d+-\d+", token):
                    expanded_tokens.append(token)
                    continue
                expanded_tokens.append(token)
                expanded_tokens.append(token.replace("-", ""))
                parts = token.split("-")
                expanded_tokens.extend(parts)
            else:
                expanded_tokens.append(token)
                singular = self._to_singular(token)
                if singular and singular != token and singular not in self.stopwords:
                    expanded_tokens.append(singular)
        return expanded_tokens

    def remove_stopwords(self, tokens: list) -> list:
        return [t for t in tokens if t not in self.stopwords]

    def preprocess(self, text: str) -> list:
        normalized = self.normalize_text(text)
        tokens = self.tokenize(normalized)
        tokens = self.remove_stopwords(tokens)
        return tokens

# ماژول ساخت سند

In [326]:
from collections import Counter

def clean_score_column(df):

    for col in ['home_score', 'away_score', 'home_penalty', 'away_penalty']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    df["score_clean"] = df["home_score"].astype(str) + "-" + df["away_score"].astype(str)

    df["has_penalty"] = (df["home_penalty"] > 0) | (df["away_penalty"] > 0)

    df["penalty_value"] = df["home_penalty"].astype(str) + "-" + df["away_penalty"].astype(str)
    return df


In [327]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter

class DocumentBuilder:
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.df = None

    def load_data(self):

        self.df = pd.read_csv(self.file_path)
        text_cols = self.df.select_dtypes(include=['object']).columns
        self.df[text_cols] = self.df[text_cols].fillna("")
        self.df = clean_score_column(self.df)

    def _parse_list_field(self, cell):
        if not cell or (isinstance(cell, float) and np.isnan(cell)):
            return []
        if isinstance(cell, str) and cell.strip().startswith('['):
            try:
                cleaned = cell.replace("&rsquor;", "'")
                return ast.literal_eval(cleaned)
            except:
                return [cell]
        if isinstance(cell, str):
            return [x.strip() for x in re.split(r'\s*\|\s*', cell) if x.strip()]
        return [str(cell)]

    def _safe_str(self, value):
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return ""
        return str(value).strip()

    def _extract_scorer_name(self, item):
        parts = [p.strip() for p in str(item).split('|')]
        if len(parts) >= 3:
            raw_name = parts[2]
            clean_name = re.sub(r'\s*\([^)]*\)', '', raw_name).strip()
            return clean_name
        return None

    def _parse_detailed_events(self, row) -> list:
        events = []
        event_cols = {
            'goal': ['home_goal', 'away_goal'],
            'goal event': ['home_goal_long', 'away_goal_long'],
            'own goal': ['home_own_goal', 'away_own_goal'],
            'penalty goal': ['home_penalty_goal', 'away_penalty_goal'],
            'penalty miss': ['home_penalty_miss_long', 'away_penalty_miss_long'],
            'penalty shootout goal': ['home_penalty_shootout_goal_long', 'away_penalty_shootout_goal_long'],
            'penalty shootout miss': ['home_penalty_shootout_miss_long', 'away_penalty_shootout_miss_long'],
            'red card': ['home_red_card', 'away_red_card'],
            'yellow red card': ['home_yellow_red_card', 'away_yellow_red_card'],
            'yellow card': ['home_yellow_card_long', 'away_yellow_card_long'],
            'substitution': ['home_substitute_in_long', 'away_substitute_in_long']
        }
        for label, cols in event_cols.items():
            for col in cols:
                for item in self._parse_list_field(row.get(col, "")):
                    if item:
                        events.append(f"{label} {item}")
        return events

    def _extract_goal_summaries(self, row):
        scorer_counts = Counter()
        assist_counts = Counter()
        players_set = set()
        players_with_team = []

        home_team = self._safe_str(row.get('home_team', ''))
        away_team = self._safe_str(row.get('away_team', ''))

        for side in ['home', 'away']:
            team_name = home_team if side == 'home' else away_team

            long_goals = self._parse_list_field(row.get(f'{side}_goal_long', ""))
            short_goals = self._parse_list_field(row.get(f'{side}_goal', ""))
            goal_events = long_goals if long_goals else short_goals
            penalty_goals = self._parse_list_field(row.get(f'{side}_penalty_goal', ""))

            for item in goal_events + penalty_goals:
                if not item:
                    continue

                scorer = self._extract_scorer_name(item)
                if scorer:
                    scorer_counts[scorer] += 1
                    players_set.add(scorer)
                    players_with_team.append(f"{scorer} ({team_name})")

                assister = self._extract_assist_name(item)
                if assister:
                    assist_counts[assister] += 1
                    players_set.add(assister)
                    players_with_team.append(f"{assister} ({team_name})")

        goal_summaries = []
        for player, goals in scorer_counts.items():
            if goals == 1:
                goal_summaries.append(f"{player} goal")
            elif goals == 2:
                goal_summaries.append(f"{player} brace")
            elif goals == 3:
                goal_summaries.append(f"{player} hattrick")
                goal_summaries.append(f"{player} hat-trick")
            elif goals == 4:
                goal_summaries.append(f"{player} poker")
                goal_summaries.append(f"{player} four goal haul")
            else:
                goal_summaries.append(f"{player} {goals} goals")

        assist_summaries = []
        for player, assists in assist_counts.items():
            if assists == 1:
                assist_summaries.append(f"{player} assist")
            elif assists == 2:
                assist_summaries.append(f"{player} 2 assists")
            else:
                assist_summaries.append(f"{player} {assists} assists")

        return (
            goal_summaries,
            assist_summaries,
            dict(scorer_counts),
            dict(assist_counts),
            list(players_set),
            list(set(players_with_team))
        )

    def _extract_scorer_name(self, item):
        if '|' in str(item):
            parts = [p.strip() for p in str(item).split('|')]
            if len(parts) >= 3:
                raw_name = parts[2]
                clean_name = re.sub(r'\s*\([^)]*\)', '', raw_name).strip()
                return clean_name

        match = re.match(r'^(.+?)\s*\(P\)\s*·\s*\d+$', str(item).strip())
        if match:
            return match.group(1).strip()

        match = re.match(r'^(.+?)\s*·\s*\d+$', str(item).strip())
        if match:
            return match.group(1).strip()

        return None

    def _extract_assist_name(self, item):
        parts = [p.strip() for p in str(item).split('|')]
        if len(parts) >= 5 and parts[3].lower().startswith('assist'):
            assist = parts[4].strip()
            if assist and assist.lower() != 'nan':
                return assist
        return None

    def build_documents(self) -> list:
        documents = []
        for idx, row in self.df.iterrows():
            # ------ events ------
            events = self._parse_detailed_events(row)

            # ------ goal_summaries ------
            (
                goal_summaries,
                assist_summaries,
                player_goal_counts,
                player_assist_counts,
                players_list,
                players_with_team
            ) = self._extract_goal_summaries(row)

            # ------ score_clean  ------
            score_value = self._safe_str(row.get('score_clean', ''))

            # ------ penalty ------
            penalty_str = ""
            try:
                h_p = int(float(row.get('home_penalty', 0) or 0))
                a_p = int(float(row.get('away_penalty', 0) or 0))
                if h_p > 0 or a_p > 0:
                    penalty_str = f"Penalty shootout {h_p}-{a_p}."
            except:
                pass

            # ------ substitution ------
            substitution_events = [
                ev for ev in events if ev.startswith('substitution')
            ]
            substitutions_str = " | ".join(substitution_events) if substitution_events else ""

            # ------ (Expected Goals) ------

            home_xg = self._safe_str(row.get('home_xg', row.get('home_xG', row.get('xG_home', ''))))
            away_xg = self._safe_str(row.get('away_xg', row.get('away_xG', row.get('xG_away', ''))))
            xg_str = ""
            if home_xg or away_xg:
                xg_str = f"Expected Goals (xG): {home_xg} - {away_xg}."


            text_body_parts = [
                f"{self._safe_str(row.get('home_team', ''))} vs {self._safe_str(row.get('away_team', ''))}.",
                f"Score {score_value}.",
                xg_str,
                penalty_str,
                f"Round {self._safe_str(row.get('Round', ''))}.",
                f"Venue {self._safe_str(row.get('Venue', ''))}.",
                f"Date {self._safe_str(row.get('Date', ''))}.",
                f"Year {self._safe_str(row.get('Year', ''))}.",
                f"Host {self._safe_str(row.get('Host', ''))}.",
                f"Referee {self._safe_str(row.get('Referee', ''))}.",
                f"Officials {self._safe_str(row.get('Officials', ''))}.",
                f"Attendance {self._safe_str(row.get('Attendance', ''))}.",
                f"Managers {self._safe_str(row.get('home_manager', ''))} and {self._safe_str(row.get('away_manager', ''))}.",
                f"Captains {self._safe_str(row.get('home_captain', ''))} and {self._safe_str(row.get('away_captain', ''))}.",
                f"Notes {self._safe_str(row.get('Notes', ''))}.",
                f"Goals {self._safe_str(row.get('home_goal', ''))} {self._safe_str(row.get('away_goal', ''))}.",
                f"Goal achievements {' | '.join(goal_summaries)}." if goal_summaries else "",
                f"Assist achievements {' | '.join(assist_summaries)}." if assist_summaries else "",
                f"Event details {' | '.join(events)}." if events else "",
                f"Substitutions: {substitutions_str}." if substitutions_str else ""
            ]

            full_text = " ".join([p for p in text_body_parts if p]).strip()

            captain_str = " ".join([
                self._safe_str(row.get('home_captain', '')),
                self._safe_str(row.get('away_captain', ''))
            ]).strip()

            players_str = " ".join(players_list) if players_list else ""
            players_with_team_str = " | ".join(players_with_team) if players_with_team else ""

            # ------ fields dictionary ------
            fields = {
                "text": full_text,
                "team": f"{self._safe_str(row.get('home_team', ''))} {self._safe_str(row.get('away_team', ''))}",
                "home_team": self._safe_str(row.get("home_team", "")),
                "away_team": self._safe_str(row.get("away_team", "")),
                "round": self._safe_str(row.get("Round", "")),
                "year": self._safe_str(row.get("Year", "")),
                "date": self._safe_str(row.get("Date", "")),
                "host": self._safe_str(row.get("Host", "")),
                "venue": self._safe_str(row.get("Venue", "")),
                "referee": self._safe_str(row.get("Referee", "")),
                "officials": self._safe_str(row.get("Officials", "")),
                "attendance": self._safe_str(row.get("Attendance", "")),
                "score": score_value,
                "home_score": self._safe_str(row.get("home_score", "")),
                "away_score": self._safe_str(row.get("away_score", "")),
                "home_penalty": self._safe_str(row.get("home_penalty", "")),
                "away_penalty": self._safe_str(row.get("away_penalty", "")),
                "home_xg": home_xg,
                "away_xg": away_xg,
                "xg": f"{home_xg} - {away_xg}" if (home_xg or away_xg) else "",
                "home_manager": self._safe_str(row.get("home_manager", "")),
                "away_manager": self._safe_str(row.get("away_manager", "")),
                "home_captain": self._safe_str(row.get("home_captain", "")),
                "away_captain": self._safe_str(row.get("away_captain", "")),
                "captain": captain_str,
                "home_goal": self._safe_str(row.get("home_goal", "")),
                "away_goal": self._safe_str(row.get("away_goal", "")),
                "home_goal_long": self._safe_str(row.get("home_goal_long", "")),
                "away_goal_long": self._safe_str(row.get("away_goal_long", "")),
                "goal_summary": " ".join(goal_summaries),
                "assist_summary": " ".join(assist_summaries),
                "player_goal_counts": str(player_goal_counts),
                "player_assist_counts": str(player_assist_counts),
                "players": players_str,
                "players_with_team": players_with_team_str,
                "substitutions": substitutions_str,
                "notes": self._safe_str(row.get("Notes", ""))
            }

            # ------ metadata ------
            metadata = {
                "home_team": self._safe_str(row.get("home_team", "")),
                "away_team": self._safe_str(row.get("away_team", "")),
                "year": self._safe_str(row.get("Year", "")),
                "venue": self._safe_str(row.get("Venue", "")),
                "score": score_value,
                "round": self._safe_str(row.get("Round", "")),
                "date": self._safe_str(row.get("Date", "")),
                "host": self._safe_str(row.get("Host", ""))
            }

            documents.append({
                "doc_id": int(idx),
                "text": full_text,
                "fields": fields,
                "metadata": metadata
            })

        return documents

# ماژول ساخت ایندکس

In [328]:
import math
from collections import defaultdict


class InvertedIndex:
    """
    Positional and field-specific Inverted Index.

    Structure:
    term -> field -> doc_id -> [positions]
    """

    def __init__(self):
        # term -> field -> doc_id -> list of positions
        self.index = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

        # doc_id -> length of main text field
        self.doc_lengths = defaultdict(int)

        # doc_id -> metadata (year, teams, etc.)
        self.doc_metadata = {}

        self.total_docs = 0

    def clear(self):
        # بازگرداندن تمام متغیرها به حالت اولیه (مشابه چیزی که در __init__ داشتید)
        self.index = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
        self.doc_lengths = defaultdict(int)
        self.doc_metadata = {}
        self.total_docs = 0
        print("Index cleared successfully.")

    def add_document(self, doc_id: int, fields: dict, preprocessor, metadata: dict = None):
        """
        Preprocesses fields and indexes tokens with positions.

        Parameters
        ----------
        doc_id : int
        fields : dict
            field_name -> text
        preprocessor : TextPreprocessor
        metadata : dict
            optional metadata such as year
        """

        self.total_docs += 1

        # store metadata for ranking (e.g. recency boost)
        if metadata is not None:
            self.doc_metadata[doc_id] = metadata

        # index fields
        for field_name, text in fields.items():

            tokens = preprocessor.preprocess(text)

            if field_name == "text":
                self.doc_lengths[doc_id] = len(tokens)

            for pos, token in enumerate(tokens):
                self.index[token][field_name][doc_id].append(pos)

    def get_postings(self, term: str, field: str = "text") -> dict:
        """
        Returns postings dictionary for a specific term and field.

        Output:
        {doc_id: [positions]}
        """
        return self.index.get(term, {}).get(field, {})

    def get_document_frequency(self, term: str, field: str = "text") -> int:
        """
        Returns number of documents containing the term
        in the specified field.
        """
        return len(self.get_postings(term, field))


#  ماژول پردازش پرس‌وجو

In [329]:
import re
from typing import List, Tuple, Set, Optional

class QueryProcessor:
    BOOLEAN_OPS = {"AND", "OR", "NOT"}
    FIELD_ALIASES = {
        "player": "players",
        "player_team": "players_with_team",
        "captain": "captain",
    }

    def __init__(self, index, preprocessor):
        self.index = index
        self.preprocessor = preprocessor

    def _map_field(self, field: str) -> str:
        return self.FIELD_ALIASES.get(field, field)

    def _parse_fielded_terms(self, query_str: str) -> List[Tuple[str, str]]:
        terms = []
        i = 0
        n = len(query_str)
        while i < n:
            while i < n and query_str[i].isspace():
                i += 1
            if i >= n:
                break
            if query_str[i] == '"':
                j = query_str.find('"', i + 1)
                if j != -1:
                    phrase = query_str[i + 1:j]
                    terms.append(("text", f'"{phrase}"'))
                    i = j + 1
                    continue
            colon = query_str.find(':', i)
            if colon != -1:
                field = query_str[i:colon]
                j = colon + 1
                while j < n and query_str[j].isspace():
                    j += 1
                if j < n and query_str[j] == '"':
                    end = query_str.find('"', j + 1)
                    if end != -1:
                        value = query_str[j + 1:end]
                        terms.append((self._map_field(field.lower()), f'"{value}"'))
                        i = end + 1
                        continue
                start = j
                while j < n:
                    if query_str[j].isspace():
                        k = j + 1
                        while k < n and query_str[k].isspace():
                            k += 1
                        if k < n:
                            m = k
                            while m < n and (query_str[m].isalnum() or query_str[m] == '_'):
                                m += 1
                            if m < n and query_str[m] == ':':
                                break
                            word = query_str[k:m]
                            if word.upper() in self.BOOLEAN_OPS:
                                break
                    j += 1
                value = query_str[start:j].strip()
                if value:
                    terms.append((self._map_field(field.lower()), value))
                i = j
                continue
            j = i
            while j < n and not query_str[j].isspace():
                j += 1
            word = query_str[i:j]
            terms.append(("text", word))
            i = j
        return terms

    def _tokenize_boolean_query(self, query_str: str) -> List[str]:
        return re.findall(r'"[^"]+"|\S+', query_str)

    def _retrieve_term(self, token: str) -> Set[int]:
        if token.startswith('"') and token.endswith('"'):
            phrase = token.strip('"')
            return self.search_phrase(phrase, field="text")
        if ":" in token:
            field, value = token.split(":", 1)
            field = self._map_field(field)
            if value.startswith('"') and value.endswith('"'):
                phrase = value.strip('"')
                return self.search_phrase(phrase, field=field)
            if " " in value or "-" in value:
                return self.search_phrase(value, field=field)
            tokens = self.preprocessor.preprocess(value)
            docs = set()
            for t in tokens:
                docs.update(self.index.get_postings(t, field).keys())
            return docs
        tokens = self.preprocessor.preprocess(token)
        docs = set()
        for t in tokens:
            docs.update(self.index.get_postings(t, "text").keys())
        return docs

    def search_boolean(self, query_str: str) -> Set[int]:
        tokens = self._tokenize_boolean_query(query_str)
        if not tokens:
            return set()
        all_docs = set(range(self.index.total_docs))
        current_set = None
        current_op = None
        for token in tokens:
            upper = token.upper()
            if upper in self.BOOLEAN_OPS:
                current_op = upper
                continue
            term_docs = self._retrieve_term(token)
            if current_set is None:
                current_set = all_docs - term_docs if current_op == "NOT" else term_docs
            else:
                if current_op == "AND":
                    current_set &= term_docs
                elif current_op == "OR":
                    current_set |= term_docs
                elif current_op == "NOT":
                    current_set -= term_docs
                else:
                    current_set &= term_docs
            current_op = None
        return current_set or set()

    def search_phrase(self, phrase: str, field: str = "text") -> Set[int]:
        field = self._map_field(field)
        tokens = self.preprocessor.preprocess(phrase)
        if not tokens:
            return set()
        first_postings = self.index.get_postings(tokens[0], field)
        candidates = set(first_postings.keys())
        results = set()
        for doc_id in candidates:
            positions = first_postings[doc_id]
            for pos in positions:
                match = True
                for offset, tok in enumerate(tokens[1:], start=1):
                    postings = self.index.get_postings(tok, field)
                    if doc_id not in postings or (pos + offset) not in postings[doc_id]:
                        match = False
                        break
                if match:
                    results.add(doc_id)
                    break
        return results

    def extract_query_terms(self, query_str: str) -> List[Tuple[str, str]]:
        tokens = self._tokenize_boolean_query(query_str)
        terms = []
        for token in tokens:
            upper = token.upper()
            if upper in self.BOOLEAN_OPS:
                continue
            if ":" in token:
                field, value = token.split(":", 1)
                field = self._map_field(field)
                if value.startswith('"') and value.endswith('"'):
                    value = value.strip('"')
                for term in self.preprocessor.preprocess(value):
                    if term:
                        terms.append((field, term))
            else:
                if token.startswith('"') and token.endswith('"'):
                    phrase = token.strip('"')
                    for term in self.preprocessor.preprocess(phrase):
                        if term:
                            terms.append(("text", term))
                else:
                    for term in self.preprocessor.preprocess(token):
                        if term:
                            terms.append(("text", term))
        return terms

    def process_query(self, query_str: str) -> Tuple[Set[int], List[Tuple[str, str]]]:
        if any(op in query_str.upper() for op in self.BOOLEAN_OPS):
            candidate_docs = self.search_boolean(query_str)
        else:
            parsed_terms = self._parse_fielded_terms(query_str)
            candidate_sets = []

            for field, term in parsed_terms:
                clean_term = term.strip('"')
                is_phrase = term.startswith('"') and term.endswith('"')
                has_hyphen = "-" in clean_term and len(clean_term) > 2

                if is_phrase or has_hyphen:
                    docs = self.search_phrase(clean_term, field)
                else:
                    tokens = self.preprocessor.preprocess(clean_term)
                    if not tokens:
                        continue
                    docs_set = None
                    for t in tokens:
                        postings = self.index.get_postings(t, field)
                        doc_ids = set(postings.keys())
                        docs_set = doc_ids if docs_set is None else (docs_set & doc_ids)
                    docs = docs_set or set()

                if docs:
                    candidate_sets.append(docs)

            candidate_docs = set.intersection(*candidate_sets) if candidate_sets else set()

        query_terms = self.extract_query_terms(query_str)
        return candidate_docs, query_terms

#  ماژول رتبه‌بندی اسناد

In [330]:
import math
from typing import List, Tuple, Set

class TFIDFRanker:


    def __init__(self, index):
        self.index = index

        self.field_weights = {
            "text": 1.0,
            "team": 3.0,
            "home_team": 2.5,
            "away_team": 2.5,
            "round": 2.5,
            "venue": 2.0,
            "referee": 1.8,
            "year": 1.0,
            "score": 2.0,
            "players": 2.8,
            "players_with_team": 2.5,
            "captain": 3.0,
            "home_captain": 2.5,
            "away_captain": 2.5,
            "home_manager": 2.0,
            "away_manager": 2.0,
            "home_goal": 2.0,
            "away_goal": 2.0,
            "home_goal_long": 2.0,
            "away_goal_long": 2.0,
            "goal_summary": 2.5,
            "assist_summary": 2.0,
            "home_xg": 1.5,
            "away_xg": 1.5,
            "xg": 2.0,
            "substitutions": 1.8,
            "notes": 1.2,
            "officials": 1.2,
            "attendance": 1.0,
            "host": 1.0,
            "date": 1.0,
        }

        if hasattr(index, 'doc_metadata') and index.doc_metadata:
            years = [
                meta.get("year") for meta in index.doc_metadata.values()
                if meta.get("year") is not None
            ]
            years = [int(y) for y in years if str(y).isdigit()]
            if years:
                self.min_year = min(years)
                self.max_year = max(years)
            else:
                self.min_year = 1930
                self.max_year = 2022
        else:
            self.min_year = 1930
            self.max_year = 2022

        self.recency_alpha = 0.15

    def _recency_factor(self, doc_id: int) -> float:

        year = self.index.doc_metadata.get(doc_id, {}).get("year")
        if year is None:
            return 1.0

        try:
            year = int(year)
        except (ValueError, TypeError):
            return 1.0

        if self.max_year == self.min_year:
            return 1.0

        normalized = (year - self.min_year) / (self.max_year - self.min_year)
        return 1.0 + self.recency_alpha * normalized

    def score_documents(
        self,
        query_terms: List[Tuple[str, str]],
        doc_candidates: Set[int]
    ) -> List[Tuple[int, float]]:

        if not doc_candidates:
            return []

        scores = {doc_id: 0.0 for doc_id in doc_candidates}

        for field, term in query_terms:

            fields_to_use = [field] if field else list(self.field_weights.keys())

            for current_field in fields_to_use:
                if current_field not in self.field_weights:
                    continue

                df = self.index.get_document_frequency(term, current_field)
                if df == 0:
                    continue

                idf = math.log10(self.index.total_docs / df)
                postings = self.index.get_postings(term, current_field)
                field_weight = self.field_weights[current_field]

                for doc_id in doc_candidates:
                    if doc_id not in postings:
                        continue

                    tf = len(postings[doc_id])

                    if current_field == "text":
                        doc_len = self.index.doc_lengths.get(doc_id, 1.0)
                        tf_norm = tf / doc_len if doc_len > 0 else 0.0
                    else:
                        tf_norm = tf

                    scores[doc_id] += field_weight * tf_norm * idf

        for doc_id in scores:
            scores[doc_id] *= self._recency_factor(doc_id)

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked

# ماژول ارزیابی

In [331]:
# src/evaluation.py

class Evaluation:
    """
    Calculates key Information Retrieval evaluation metrics (MAP, MRR, Recall@k).
    """

    @staticmethod
    def precision_at_k(retrieved: list, relevant: set, k: int) -> float:
        """
        Precision at rank k: proportion of relevant documents in the top-k results.
        """
        if not retrieved or k <= 0:
            return 0.0
        retrieved_k = retrieved[:k]
        hits = len([doc for doc in retrieved_k if doc in relevant])
        return hits / k

    @staticmethod
    def average_precision(retrieved: list, relevant: set) -> float:
        """
        Average Precision: mean of precision values at each relevant document position.
        Returns 0.0 if there are no relevant documents.
        """
        if not relevant:
            return 0.0

        ap = 0.0
        hits = 0
        for idx, doc in enumerate(retrieved):
            if doc in relevant:
                hits += 1
                ap += hits / (idx + 1)
        return ap / len(relevant)

    @staticmethod
    def reciprocal_rank(retrieved: list, relevant: set) -> float:
        """
        Reciprocal Rank: 1 / rank of the first relevant document (0 if none).
        """
        for idx, doc in enumerate(retrieved):
            if doc in relevant:
                return 1.0 / (idx + 1)
        return 0.0

    @staticmethod
    def recall_at_k(retrieved: list, relevant: set, k: int) -> float:
        """
        Recall@k: proportion of all relevant documents that are retrieved in the top-k results.
        Returns 0.0 if there are no relevant documents.
        """
        if not relevant:
            return 0.0

        retrieved_k = retrieved[:k]
        hits = len([doc for doc in retrieved_k if doc in relevant])
        return hits / len(relevant)

In [332]:
import pandas as pd
import re
import ast

def build_relevance_sets(csv_file):
    df = pd.read_csv(csv_file)
    df = clean_score_column(df)
    relevance = {}
    def parse_list_field(cell):
        if not cell or (isinstance(cell, float) and pd.isna(cell)):
            return []
        if isinstance(cell, str) and cell.strip().startswith('['):
            try:
                cleaned = cell.replace("&rsquor;", "'")
                return ast.literal_eval(cleaned)
            except:
                return [cell]
        if isinstance(cell, str):
            return [x.strip() for x in re.split(r'\s*\|\s*', cell) if x.strip()]
        return [str(cell)]

    def extract_scorer_name(item):
        if '|' in str(item):
            parts = [p.strip() for p in str(item).split('|')]
            if len(parts) >= 3:
                raw_name = parts[2]
                clean_name = re.sub(r'\s*\([^)]*\)', '', raw_name).strip()
                return clean_name

        match = re.match(r'^(.+?)\s*\(P\)\s*·\s*\d+$', str(item).strip())
        if match:
            return match.group(1).strip()

        match = re.match(r'^(.+?)\s*·\s*\d+$', str(item).strip())
        if match:
            return match.group(1).strip()

        return None

    def extract_assist_name(item):
        parts = [p.strip() for p in str(item).split('|')]
        if len(parts) >= 5 and parts[3].lower().startswith('assist'):
            assist = parts[4].strip()
            if assist and assist.lower() != 'nan':
                return assist
        return None


    # 1
    relevance["messi goal final"] = set(
        df[(df["Round"] == "Final") &  # فقط فینال اصلی، نه Quarter-final
           ((df["home_goal_long"].str.contains("Messi", case=False, na=False)) |
            (df["away_goal_long"].str.contains("Messi", case=False, na=False)))].index
    )

    # 2
    mbappe_assist = set()
    for idx, row in df.iterrows():
        for col in ['home_goal_long', 'away_goal_long', 'home_goal', 'away_goal']:
            if col not in df.columns:
                continue
            for item in parse_list_field(row.get(col, "")):
                assister = extract_assist_name(item)
                if assister and 'mbapp' in assister.lower():
                    mbappe_assist.add(idx)
    relevance["mbappe assist"] = mbappe_assist

    # 3
    relevance["croatia third-place"] = set(
        df[(df["Round"].str.contains("Third", case=False, na=False)) &
           ((df["home_team"].str.contains("Croatia", case=False, na=False)) |
            (df["away_team"].str.contains("Croatia", case=False, na=False)))].index
    )

    # 4
    relevance["morocco semi-final"] = set(
        df[(df["Round"].str.contains("Semi", case=False, na=False)) &
           ((df["home_team"].str.contains("Morocco", case=False, na=False)) |
            (df["away_team"].str.contains("Morocco", case=False, na=False)))].index
    )

    # 5
    relevance['referee:"Szymon Marciniak"'] = set(
        df[df["Referee"].str.contains("Marciniak", case=False, na=False)].index
    )

    # 6
    relevance['score:"3-3"'] = set(
        df[df["score_clean"] == "3-3"].index
    )


    # 7
    relevance['venue:"Lusail Iconic Stadium"'] = set(
        df[df["Venue"].str.contains("Lusail", case=False, na=False)].index
    )

    # 8
    relevance["penalty shootout"] = set(
        df[df["Notes"].astype(str).str.contains("penalty", case=False, na=False)].index
    )

    # 9
    relevance["team:Brazil"] = set(
        df[(df["home_team"].str.contains("Brazil", case=False, na=False)) |
           (df["away_team"].str.contains("Brazil", case=False, na=False))].index
    )

    # 10
    relevance["yellow card"] = set(
        df[(df["home_yellow_card_long"].notna()) |
           (df["away_yellow_card_long"].notna())].index
    )

    return relevance


In [333]:
# src/main.py

import os

def main():
    csv_file = "matches_1930_2022.csv"
    if not os.path.exists(csv_file):
        print(f"Dataset not found at {csv_file}")
        return

    print("--- Step 1 & 2: Loading Data & Building/Preprocessing Documents ---")

    builder = DocumentBuilder(csv_file)
    builder.load_data()

    docs = builder.build_documents()
    print(f"Loaded {len(docs)} match documents.\n")

    preprocessor = TextPreprocessor()

    index = InvertedIndex()



    print("--- Step 3: Building Inverted Index ---")

    for d in docs:
        index.add_document(
            d["doc_id"],
            d["fields"],
            preprocessor,
            metadata=d["metadata"]
        )

    print("Inverted Index built successfully.\n")


    query_proc = QueryProcessor(index, preprocessor)
    ranker = TFIDFRanker(index)

    # -----------------------------------------------------
    #  process_query
    # -----------------------------------------------------
    def run_query(query_text: str):

        candidate_docs, query_terms = query_proc.process_query(query_text)
        if not candidate_docs:
            return []
        return ranker.score_documents(query_terms, candidate_docs)
    # -----------------------------------------------------
    # Sample Queries
    # -----------------------------------------------------
    print("--- Step 4 & 5: Sample Searches ---")

    sample_queries = [
        "player_team Kylian Mbappe (France)",
        "substitution Marcus Thuram",
        "messi",
        "mbappe",
        "messi AND mbappe",
        "messi OR mbappe",
        "messi NOT mbappe",
        "mbappe goal",
        "Messi goal",
        "Messi's goal ",
        "mbappe hat-trick",
        "Bartlett brace",
        "penalty AND quarter-finals",
        "team:Argentina round:Final",
        "referee:”Szymon Marciniak”",
        "mbappe assist",
        "final penalty",
        "own goal",
        "extra time goal",
        "yellow cards argentina",
        "substitutions france",
        "matches in lusail",
        "referee marciniak",
        "captain luka modric",
        "penalties miss",
        "red card",
        "0-0 draw",
        "team:Portugal player:Goncalo Ramos",
        "round:Semi-finals team:Morocco",
        "messi final",
        "team:Argentina round:Final",
        'referee:"Szymon Marciniak"'
    ]

    for sq in sample_queries:

        results = run_query(sq)

        print(f"Query: '{sq}'")
        print(f"Results Count: {len(results)}")

        for i, (doc_id, score) in enumerate(results[:3]):

            if doc_id >= len(docs):
                continue

            doc_info = docs[doc_id]
            meta = doc_info["metadata"]

            print(
                f"  Rank {i+1} | DocID: {doc_id} | "
                f"{meta['home_team']} vs {meta['away_team']} "
                f"({meta['year']}) | Score: {score:.4f}"
            )

        print("-" * 50)

    # -----------------------------------------------------
    # Evaluation
    # -----------------------------------------------------
    print("\n--- Step 6: Evaluation ---")
    eval_queries = build_relevance_sets(csv_file)

    ap_scores = []
    rr_scores = []
    recall_100_scores = []

    for q_text, relevant_set in eval_queries.items():
        results = run_query(q_text)
        retrieved_ids = [doc_id for doc_id, _ in results]

        ap = Evaluation.average_precision(retrieved_ids, relevant_set)
        rr = Evaluation.reciprocal_rank(retrieved_ids, relevant_set)
        recall_100 = Evaluation.recall_at_k(retrieved_ids, relevant_set, k=100)

        ap_scores.append(ap)
        rr_scores.append(rr)
        recall_100_scores.append(recall_100)

        print(f"Query: '{q_text}' -> AP: {ap:.4f}, RR: {rr:.4f}, R@100: {recall_100:.4f}")


    if ap_scores:
        mean_ap = sum(ap_scores) / len(ap_scores)
    else:
        mean_ap = 0.0

    if rr_scores:
        mean_rr = sum(rr_scores) / len(rr_scores)
    else:
        mean_rr = 0.0

    if recall_100_scores:
        mean_recall_100 = sum(recall_100_scores) / len(recall_100_scores)
    else:
        mean_recall_100 = 0.0

    print(f"\nOverall MAP: {mean_ap:.4f}")
    print(f"Overall MRR: {mean_rr:.4f}")
    print(f"Overall Recall@100: {mean_recall_100:.4f}")

if __name__ == "__main__":
    main()


--- Step 1 & 2: Loading Data & Building/Preprocessing Documents ---
Loaded 964 match documents.

--- Step 3: Building Inverted Index ---
Inverted Index built successfully.

--- Step 4 & 5: Sample Searches ---
Query: 'player_team Kylian Mbappe (France)'
Results Count: 11
  Rank 1 | DocID: 106 | France vs Peru (2018) | Score: 0.1266
  Rank 2 | DocID: 12 | France vs Poland (2022) | Score: 0.1180
  Rank 3 | DocID: 42 | France vs Denmark (2022) | Score: 0.1067
--------------------------------------------------
Query: 'substitution Marcus Thuram'
Results Count: 5
  Rank 1 | DocID: 12 | France vs Poland (2022) | Score: 0.0597
  Rank 2 | DocID: 0 | Argentina vs France (2022) | Score: 0.0587
  Rank 3 | DocID: 2 | France vs Morocco (2022) | Score: 0.0505
--------------------------------------------------
Query: 'messi'
Results Count: 19
  Rank 1 | DocID: 149 | Nigeria vs Argentina (2014) | Score: 0.0706
  Rank 2 | DocID: 91 | Nigeria vs Argentina (2018) | Score: 0.0438
  Rank 3 | DocID: 163 | Ar